In [1]:
# Change working directory
from pathlib import Path
import os

# Add path to SWMM file reports
path = Path(r"P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\ShawRd_pump\Q3_50P")

os.chdir(path)

print(Path.cwd())

P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\ShawRd_pump\Q3_50P


In [2]:
# Load python libraries
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

In [3]:
# Load pre-project report
pre_rpt = read_rpt_file("SSF_SDMP_ShawRd_pump_pre.rpt")

# Extract data
pre = pre_rpt.node_flooding_summary.copy()

# Ensure Node is index
pre.index.name = "Node"

# Rename columns
pre = pre.rename(columns={
    "Hours_Flooded": "Pre Hours Flooded",
    "Total_Flood_Volume_10^6 gal": "Pre Total Flood Vol (MG)",
})

# Keep only needed columns
pre = pre[[
    "Pre Hours Flooded",
    "Pre Total Flood Vol (MG)"
]]

pre.head()

,Pre Hours Flooded,Pre Total Flood Vol (MG)
Node,,
sQ2106,3.77,0.027
sQ2107,4.81,0.047
sQ2108,4.20,0.025
sQ2108A,4.35,0.026
sQ2109,4.57,0.032


In [4]:
# Load post-project report
post_rpt = read_rpt_file("SSF_SDMP_ShawRd_pump_imp2.rpt")

# If no flooding occurred, create an empty post-project table
if post_rpt.node_flooding_summary is None:
    print("No node flooding occurred in the post-project model.")

    post = pd.DataFrame(columns=[
        "Post Hours Flooded",
        "Post Total Flood Vol (MG)"
    ])

    post.index.name = "Node"

else:
    # Extract data
    post = post_rpt.node_flooding_summary.copy()

    # Ensure Node is index
    post.index.name = "Node"

    # Rename columns
    post = post.rename(columns={
        "Hours_Flooded": "Post Hours Flooded",
        "Total_Flood_Volume_10^6 gal": "Post Total Flood Vol (MG)",
    })

    # Keep only needed columns
    post = post[[
        "Post Hours Flooded",
        "Post Total Flood Vol (MG)"
    ]]

post.head()

,Post Hours Flooded,Post Total Flood Vol (MG)
Node,,
sQ2107,1.75,0.003
sQ2109,1.74,0.003
sQ2109A,0.01,0.000
ShawRdPumpStation,1.49,0.095


In [5]:
# ============================================================
# MERGE PRE- AND POST-PROJECT FLOODING RESULTS
# ============================================================

# Outer join preserves:
# - Nodes flooding only in pre-project
# - Nodes flooding only in post-project
# - Nodes flooding in both scenarios

comparison = pre.join(post, how="outer")

comparison.head()

,Pre Hours Flooded,Pre Total Flood Vol (MG),Post Hours Flooded,Post Total Flood Vol (MG)
Node,,,,
ShawRdPumpStation,3.26,0.350,1.49,0.095
sQ2106,3.77,0.027,NaN,NaN
sQ2107,4.81,0.047,1.75,0.003
sQ2108,4.20,0.025,NaN,NaN
sQ2108A,4.35,0.026,NaN,NaN


In [6]:
# ============================================================
# REPLACE MISSING FLOODING VALUES WITH ZERO
#
# Missing values indicate the node did not flood in that
# scenario.
# ============================================================

fill_cols = [
    "Pre Hours Flooded",
    "Post Hours Flooded",
    "Pre Total Flood Vol (MG)",
    "Post Total Flood Vol (MG)"
]

for col in fill_cols:
    if col in comparison.columns:
        comparison[col] = comparison[col].fillna(0)

comparison.head()

,Pre Hours Flooded,Pre Total Flood Vol (MG),Post Hours Flooded,Post Total Flood Vol (MG)
Node,,,,
ShawRdPumpStation,3.26,0.350,1.49,0.095
sQ2106,3.77,0.027,0.00,0.000
sQ2107,4.81,0.047,1.75,0.003
sQ2108,4.20,0.025,0.00,0.000
sQ2108A,4.35,0.026,0.00,0.000


In [7]:
# ============================================================
# FLOOD VOLUME REDUCTION
# ============================================================

comparison["Total Flood Vol Reduction (MG)"] = (
    comparison["Pre Total Flood Vol (MG)"]
    - comparison["Post Total Flood Vol (MG)"]
)

comparison["Total Flood Vol Percent Reduction"] = (
    comparison["Total Flood Vol Reduction (MG)"]
    / comparison["Pre Total Flood Vol (MG)"].where(
        comparison["Pre Total Flood Vol (MG)"] > 0
    )
    * 100
).fillna(0)


# ============================================================
# FLOOD DURATION REDUCTION
# ============================================================

comparison["Hours Flooded Reduction"] = (
    comparison["Pre Hours Flooded"]
    - comparison["Post Hours Flooded"]
)

comparison["Hours Flooded Percent Reduction"] = (
    comparison["Hours Flooded Reduction"]
    / comparison["Pre Hours Flooded"].where(
        comparison["Pre Hours Flooded"] > 0
    )
    * 100
).fillna(0)


# ============================================================
# ROUND RESULTS FOR REPORTING
# ============================================================

comparison = comparison.round({
    "Pre Hours Flooded": 2,
    "Post Hours Flooded": 2,
    "Pre Total Flood Vol (MG)": 3,
    "Post Total Flood Vol (MG)": 3,
    "Total Flood Vol Reduction (MG)": 3,
    "Total Flood Vol Percent Reduction": 1,
    "Hours Flooded Reduction": 2,
    "Hours Flooded Percent Reduction": 1
})


# ============================================================
# SORT BY PRE-PROJECT FLOOD SEVERITY
#
# Largest pre-project flood volumes first.
# ============================================================

comparison = comparison.sort_values(
    by="Pre Total Flood Vol (MG)",
    ascending=False
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

comparison.head(5)

,Pre Hours Flooded,Pre Total Flood Vol (MG),Post Hours Flooded,Post Total Flood Vol (MG),Total Flood Vol Reduction (MG),Total Flood Vol Percent Reduction,Hours Flooded Reduction,Hours Flooded Percent Reduction
Node,,,,,,,,
ShawRdPumpStation,3.26,0.350,1.49,0.095,0.255,72.9,1.77,54.3
sQ2107,4.81,0.047,1.75,0.003,0.044,93.6,3.06,63.6
sQ2109,4.57,0.032,1.74,0.003,0.029,90.6,2.83,61.9
sQ2106,3.77,0.027,0.00,0.000,0.027,100.0,3.77,100.0
sQ2108A,4.35,0.026,0.00,0.000,0.026,100.0,4.35,100.0


In [8]:
# ============================================================
# FLOOD PERFORMANCE SUMMARY
#
# Purpose:
# Summarize pre- and post-project flooding performance for
# SDMP project ranking.
#
# Scoring Metrics:
#   1. Total Flood Volume (gal)
#   2. Average Node Flood Duration (hr)
#   3. Percent Network Flooding (%)
#
# Notes:
# - Average duration is calculated only for nodes that flooded
#   in the pre-project condition.
# - Nodes that flooded pre-project but no longer flood
#   post-project are counted as 0 hr post-project flooding.
# ============================================================


# ============================================================
# EXCLUDE OUTFALLS FROM NETWORK FLOODING ANALYSIS
# ============================================================

# Get non-outfall nodes from the pre-project node depth summary
non_outfall_nodes = pre_rpt.node_depth_summary[
    pre_rpt.node_depth_summary["Type"].str.upper() != "OUTFALL"
].index

# Filter comparison table to only non-outfall nodes
network_comparison = comparison.loc[
    comparison.index.isin(non_outfall_nodes)
].copy()


# ============================================================
# FLOODED NODE COUNTS
# ============================================================

pre_flooded_nodes = (network_comparison["Pre Hours Flooded"] > 0).sum()
post_flooded_nodes = (network_comparison["Post Hours Flooded"] > 0).sum()

flooded_nodes_reduction = pre_flooded_nodes - post_flooded_nodes

flooded_nodes_percent_reduction = (
    flooded_nodes_reduction / pre_flooded_nodes * 100
    if pre_flooded_nodes > 0 else 0
)


# ============================================================
# PERCENT NETWORK FLOODING
#
# Uses total non-outfall nodes from the pre-project report node depth summary.
# ============================================================

total_nodes = len(non_outfall_nodes)

pre_pct_nodes_flooded = (
    pre_flooded_nodes / total_nodes * 100
    if total_nodes > 0 else 0
)

post_pct_nodes_flooded = (
    post_flooded_nodes / total_nodes * 100
    if total_nodes > 0 else 0
)

pct_nodes_flooded_reduction = (
    pre_pct_nodes_flooded - post_pct_nodes_flooded
)


# ============================================================
# TOTAL FLOOD VOLUME
#
# SWMM reports flood volume in million gallons (MG).
# Convert to gallons for consistency with scoring matrix.
# ============================================================

network_pre_total_mg = network_comparison["Pre Total Flood Vol (MG)"].sum()
network_post_total_mg = network_comparison["Post Total Flood Vol (MG)"].sum()

network_pre_total_gal = network_pre_total_mg * 1_000_000
network_post_total_gal = network_post_total_mg * 1_000_000

network_reduction_gal = network_pre_total_gal - network_post_total_gal

network_percent_reduction = (
    network_reduction_gal / network_pre_total_gal * 100
    if network_pre_total_gal > 0 else 0
)


# ============================================================
# AVERAGE NODE FLOOD DURATION
#
# Calculated separately for flooded nodes in each condition.
# - Pre-project average uses nodes that flooded pre-project.
# - Post-project average uses nodes that flooded post-project.
#
# Flooding extent / migration is captured separately by the
# Percent Network Flooding metric.
# ============================================================

pre_flooded_mask = network_comparison["Pre Hours Flooded"] > 0
post_flooded_mask = network_comparison["Post Hours Flooded"] > 0

pre_avg_duration = network_comparison.loc[
    pre_flooded_mask,
    "Pre Hours Flooded"
].mean()

post_avg_duration = network_comparison.loc[
    post_flooded_mask,
    "Post Hours Flooded"
].mean()

pre_avg_duration = 0 if pd.isna(pre_avg_duration) else pre_avg_duration
post_avg_duration = 0 if pd.isna(post_avg_duration) else post_avg_duration

avg_duration_reduction = pre_avg_duration - post_avg_duration

avg_duration_percent_reduction = (
    avg_duration_reduction / pre_avg_duration * 100
    if pre_avg_duration > 0 else 0
)

# ============================================================
# RESULTS
# ============================================================

print("\n====================================================")
print("NETWORK FLOODING PERFORMANCE SUMMARY")
print("====================================================")

print("\n--- Total Flood Volume ---")
print(f"Pre-Project Total Flood Volume: {network_pre_total_gal:,.0f} gal")
print(f"Post-Project Total Flood Volume: {network_post_total_gal:,.0f} gal")
print(f"Flood Volume Reduction: {network_reduction_gal:,.0f} gal")
print(f"Flood Volume Percent Reduction: {network_percent_reduction:.1f}%")

print("\n--- Average Flood Duration of Flooded Nodes ---")
print(f"Pre-Project Average Flood Duration: {pre_avg_duration:.2f} hr")
print(f"Post-Project Average Flood Duration: {post_avg_duration:.2f} hr")
print(f"Average Flood Duration Reduction: {avg_duration_reduction:.2f} hr")
print(f"Average Flood Duration Percent Reduction: {avg_duration_percent_reduction:.1f}%")

print("\n--- Percent Network Flooding ---")
print(f"Total Nodes in Network: {total_nodes}")
print(f"Pre-Project Flooded Nodes: {pre_flooded_nodes}")
print(f"Post-Project Flooded Nodes: {post_flooded_nodes}")
print(f"Pre-Project Percent Network Flooding: {pre_pct_nodes_flooded:.1f}%")
print(f"Post-Project Percent Network Flooding: {post_pct_nodes_flooded:.1f}%")
print(f"Percent Network Flooding Reduction: {pct_nodes_flooded_reduction:.1f}%")


NETWORK FLOODING PERFORMANCE SUMMARY

--- Total Flood Volume ---
Pre-Project Total Flood Volume: 556,000 gal
Post-Project Total Flood Volume: 101,000 gal
Flood Volume Reduction: 455,000 gal
Flood Volume Percent Reduction: 81.8%

--- Average Flood Duration of Flooded Nodes ---
Pre-Project Average Flood Duration: 4.13 hr
Post-Project Average Flood Duration: 1.25 hr
Average Flood Duration Reduction: 2.88 hr
Average Flood Duration Percent Reduction: 69.8%

--- Percent Network Flooding ---
Total Nodes in Network: 8
Pre-Project Flooded Nodes: 8
Post-Project Flooded Nodes: 4
Pre-Project Percent Network Flooding: 100.0%
Post-Project Percent Network Flooding: 50.0%
Percent Network Flooding Reduction: 50.0%
